In [3]:
from astropy.io import fits
import os
import glob
import re
from astropy.visualization import make_lupton_rgb
from astropy.visualization import PercentileInterval
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.wcs import WCS, FITSFixedWarning
from astropy.coordinates import SkyCoord
import shutil
from scipy.ndimage import rotate
import astropy.units as u
import warnings

# These warnings just note that astropy filled in derived WCS
# keywords (DATE-*, OBSGEO-*) from the FITS header -- harmless here.
warnings.filterwarnings("ignore", category=FITSFixedWarning)

In [4]:
SOURCES = [
    # PaA sources: pages already exist on the site from the old (now-removed) flat
    # GEKO_RUN_DIRECTORY layout, so they're commented out here rather than deleted --
    # leaving them out of SOURCES means process_source() is never called for them, so
    # their existing sources/<ID>/ pages + homepage cards are left completely alone.
    # Uncomment once you've re-run geko on them and moved the results under
    # GEKO_BATCH_ROOTS (batchN/<ID>/...) like the Halpha sources below -- until then,
    # find_geko_pngs() has nowhere to find their images anymore.
    # {"id": 16887, "filter": "F444W", "line": "PaA"},
    # {"id": 21636, "filter": "F444W", "line": "PaA"},
    # {"id": 15665, "filter": "F444W", "line": "PaA"},
    # {"id": 8910, "filter": "F444W", "line": "PaA"},
    # {"id": 6170, "filter": "F444W", "line": "PaA"},
    # {"id": 3700, "filter": "F444W", "line": "PaA"},

    # Halpha clean-env batch (see GEKO_BATCH_ROOTS below) -- batch4, the latest
    # complete rerun of this source list as of 2026-07.
    {"id": 3494, "filter": "F444W", "line": "Ha"},
    {"id": 3976, "filter": "F444W", "line": "Ha"},
    {"id": 4158, "filter": "F444W", "line": "Ha"},
    {"id": 4831, "filter": "F444W", "line": "Ha"},
    {"id": 6084, "filter": "F444W", "line": "Ha"},
    {"id": 11406, "filter": "F444W", "line": "Ha"},
    {"id": 11733, "filter": "F444W", "line": "Ha"},
    {"id": 12086, "filter": "F444W", "line": "Ha"},
    {"id": 12277, "filter": "F444W", "line": "Ha"},
    {"id": 12722, "filter": "F444W", "line": "Ha"},
    {"id": 14666, "filter": "F444W", "line": "Ha"},
    {"id": 15938, "filter": "F444W", "line": "Ha"},
    {"id": 15941, "filter": "F444W", "line": "Ha"},
    {"id": 18212, "filter": "F444W", "line": "Ha"},
    {"id": 18944, "filter": "F444W", "line": "Ha"},
    {"id": 20219, "filter": "F444W", "line": "Ha"},
    {"id": 22835, "filter": "F444W", "line": "Ha"},
    {"id": 23955, "filter": "F444W", "line": "Ha"},
    {"id": 30563, "filter": "F444W", "line": "Ha"},
    {"id": 34403, "filter": "F444W", "line": "Ha"},
    {"id": 36154, "filter": "F444W", "line": "Ha"},
]

# Field/grism orientation angle -- same for every source in this
# pointing, matches image_generation.ipynb.
ROT_ANGLE = -35.74

size = (40, 40)

filters = {
    "blue":  "F200W",
    "green": "F277W",
    "red":   "F444W"
}

BASE_EDR_DATA_DIRECTORY = "/xdisk/egami/aakhtarkavan/edr_products/"

SPEC_PDF_DIRECTORY = (
    "/xdisk/egami/aakhtarkavan/"
    "edr_products/sapphires_edr_spec_pdf"
)

GRISM_OVERVIEW_DIRECTORY = (
    "/home/u7/aakhtarkavan/git_projects/"
    "Galaxy_Kinematics/notebooks/all_sources_grism_imaging"
)

# batchN layout (clean-env Halpha batches, moved to xdisk to save home-dir storage):
# {root}/batchN/{ID}/{ID}Ha/{ID}_*.png -- add more roots here if batches ever get split
# across locations. New batches (batch5, batch6, ...) are picked up automatically by
# find_geko_pngs()/find_latest_batch_run_dir() below -- no edits needed here, just make
# sure the batchN directory lives under one of these roots.
GEKO_BATCH_ROOTS = [
    "/xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissions",
]

SOURCES_BASE_DIRECTORY = (
    "/home/u7/aakhtarkavan/git_projects/"
    "Galaxy_Kinematics/sources"
)

INDEX_PATH = "../index.html"

In [5]:
# Load the full spectroscopic catalog once; individual rows are
# looked up per-source inside process_source().
spec_cat = fits.getdata(
    f"{BASE_EDR_DATA_DIRECTORY}"
    "sapphires_edr_catalogs/"
    "sapphires_edr_spec_cat.fits"
)

In [6]:
# MAKE CUTOUT FUNCTION
def make_rotated_cutout(filter_name, size, coord, pad_factor=1.6):

    sci_file = (
        f"{BASE_EDR_DATA_DIRECTORY}"
        f"sapphires_edr_nircam_sci/"
        f"4750_{filter_name}_v05_sci.fits"
    )

    hdu = fits.open(sci_file)

    data = hdu['SCI'].data

    wcs = WCS(hdu['SCI'].header)

    # Cut out a larger box than needed so that after rotation
    # there are no zero-padded corners within the final frame.
    pad_size = (
        int(np.ceil(size[0] * pad_factor)),
        int(np.ceil(size[1] * pad_factor)),
    )

    cutout = Cutout2D(
        data,
        position=coord,
        size=pad_size,
        wcs=wcs
    )

    rot = rotate(
        cutout.data,
        angle=ROT_ANGLE,
        reshape=False,
        order=3,
        cval=0.0
    )

    rot = np.nan_to_num(rot)

    # Crop back down to the requested size from the center, now
    # that the rotated edges are clean.
    cy, cx = rot.shape[0] // 2, rot.shape[1] // 2
    half_h, half_w = size[0] // 2, size[1] // 2

    rot = rot[
        cy - half_h: cy + half_h,
        cx - half_w: cx + half_w
    ]

    return rot


# COLOR COMBINATION (asinh stretch, Lupton et al. 2004)
# This is the same scheme used for most published JWST/HST color
# images: a single nonlinear (asinh) stretch is applied to all
# three channels together so that color ratios are preserved from
# faint outskirts to bright cores, instead of each channel being
# clipped independently (which washes bright regions out to white).
def channel_minimum(img, percentile=1.0):
    return np.nanpercentile(img, percentile)


def make_rgb_cutout(coord):

    blue  = make_rotated_cutout(filters["blue"], size, coord)
    green = make_rotated_cutout(filters["green"], size, coord)
    red   = make_rotated_cutout(filters["red"], size, coord)

    minimum = [
        channel_minimum(red),
        channel_minimum(green),
        channel_minimum(blue),
    ]

    # stretch sets the linear scale before the asinh nonlinearity --
    # smaller values boost faint flux more. Q sets how smoothly the
    # bright end rolls off instead of clipping to white.
    return make_lupton_rgb(
        red, green, blue,
        minimum=minimum,
        stretch=0.4,
        Q=8
    )

In [7]:
# GEKO PNG lookup -- batchN layout under GEKO_BATCH_ROOTS (see config cell above).
def find_latest_batch_run_dir(ID):
    """Return the {ID}/ run directory from the highest-numbered batchN that has this
    source under any GEKO_BATCH_ROOTS entry, so a resubmitted/fixed rerun in a later
    batch always wins over an older one. None if ID isn't in any batchN there."""
    candidates = {}
    for root in GEKO_BATCH_ROOTS:
        for batch_dir in glob.glob(f"{root}/batch*/{ID}"):
            m = re.search(r"batch(\d+)", batch_dir)
            if m:
                candidates[int(m.group(1))] = batch_dir
    if not candidates:
        return None
    return candidates[max(candidates)]


def find_geko_pngs(ID):
    """Every {ID}_*.png for this source: the newest batchN/{ID}/**/*.png match (Halpha
    clean-env batches, nested under an {ID}Ha/ subdirectory). Empty list if ID isn't in
    any batchN under GEKO_BATCH_ROOTS (e.g. the PaA sources -- see SOURCES above)."""
    run_dir = find_latest_batch_run_dir(ID)
    if run_dir is None:
        return []
    return sorted(glob.glob(f"{run_dir}/**/{ID}_*.png", recursive=True))

In [ ]:
def process_source(source):

    ID = source["id"]
    FILTER = source["filter"]
    line = source["line"]

    SOURCE_DIRECTORY = f"{SOURCES_BASE_DIRECTORY}/{ID}"
    os.makedirs(SOURCE_DIRECTORY, exist_ok=True)

    row = spec_cat[spec_cat['ID'] == ID][0]

    coord = SkyCoord(
        ra=row['RA']*u.deg,
        dec=row['DEC']*u.deg
    )

    zspec = row['zspec']

    # RGB CUTOUT
    rgb_cube = make_rgb_cutout(coord)

    plt.figure(figsize=(6,6))

    plt.imshow(
        rgb_cube,
        origin='lower'
    )

    plt.axis('off')

    plt.savefig(
        f"{SOURCE_DIRECTORY}/{ID}_rgb_rot.png",
        dpi=300,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close()

    # SPECTRA + GRISM OVERVIEW
    r_pdf = (
        f"{SPEC_PDF_DIRECTORY}/"
        f"spec_2d_M0416_{FILTER}_ID{ID}_R.pdf"
    )

    c_pdf = (
        f"{SPEC_PDF_DIRECTORY}/"
        f"spec_2d_M0416_{FILTER}_ID{ID}_C.pdf"
    )

    shutil.copy(r_pdf, SOURCE_DIRECTORY)
    shutil.copy(c_pdf, SOURCE_DIRECTORY)

    grism_file = (
        f"{GRISM_OVERVIEW_DIRECTORY}/"
        f"{ID}_{FILTER}_{line}_grism_overview.png"
    )

    shutil.copy(grism_file, SOURCE_DIRECTORY)

    # GEKO -- copy every diagnostic PNG for this source, not just the summary plot, so
    # the GEKO subpage can show all of them. find_geko_pngs() looks up the newest
    # batchN match under GEKO_BATCH_ROOTS (see helper cell above) -- empty for any
    # source not yet in a batchN there (e.g. PaA sources, currently commented out of
    # SOURCES for exactly this reason).
    geko_files = find_geko_pngs(ID)

    for geko_file in geko_files:
        shutil.copy(geko_file, SOURCE_DIRECTORY)

    geko_image_names = [os.path.basename(f) for f in geko_files]
    has_geko = bool(geko_image_names)
    has_dingo = os.path.exists(f"{SOURCE_DIRECTORY}/dingo_fit.png")

    if geko_image_names:

        geko_gallery = "\n".join(
            f"""
        <div class="col-md-6 mb-4">
            <img src="{name}" class="img-fluid rounded shadow">
        </div>"""
            for name in geko_image_names
        )

        geko_page_html = f"""
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <title>Source {ID} - GEKO Results</title>

    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">

</head>

<body class="bg-dark text-light">

<div class="container py-5">

    <a href="index.html"
       class="btn btn-secondary mb-4">
        ← Back to Source {ID}
    </a>

    <h1 class="mb-4">
        Source {ID} - GEKO Results
    </h1>

    <div class="row g-4">
        {geko_gallery}
    </div>

</div>

</body>

</html>
"""

        with open(f"{SOURCE_DIRECTORY}/geko.html", "w") as f:
            f.write(geko_page_html)

    # HTML
    geko_block = (
        f'<a href="geko.html">'
        f'<img src="{ID}_summary.png" class="img-fluid rounded shadow">'
        f'</a>'
        if has_geko
        else "GEKO results coming soon"
    )

    dingo_block = (
        '<img src="dingo_fit.png" class="img-fluid rounded shadow">'
        if has_dingo
        else "DINGO results coming soon"
    )

    html = f"""
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <title>Source {ID}</title>

    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">

</head>

<body class="bg-dark text-light">

<div class="container py-5">

    <!-- BACK BUTTON -->

    <a href="../../index.html"
       class="btn btn-secondary mb-4">
        ← Back
    </a>

    <!-- TOP ROW -->

    <div class="row align-items-start mb-5">

        <!-- LEFT SIDE -->

        <div class="col-md-8">

            <h1 class="mb-1">
                Source {ID}
            </h1>

            <p class="text-secondary fs-5 mb-2">
                z = {zspec:.3f}
            </p>

            <p class="mb-1">
                Filter: {FILTER}
            </p>

            <p class="mb-0">
                Line: {line}
            </p>

        </div>

        <!-- RIGHT SIDE -->

        <div class="col-md-4 text-end">

            <img src="{ID}_rgb_rot.png"
                 class="img-fluid rounded shadow"
                 style="max-height: 250px;">

        </div>

    </div>

    <!-- OBSERVATIONS -->

    <h2 class="mb-4">
        Observations
    </h2>

    <div class="mb-5 text-center">

        <img src="{ID}_{FILTER}_{line}_grism_overview.png"
             class="img-fluid rounded shadow">

    </div>

    <!-- SPECTRA -->

    <h2 class="mb-4">
        Spectra
    </h2>

    <div class="row g-4 mb-5">

        <!-- R SPECTRUM -->

        <div class="col-md-6">

            <h5 class="text-center mb-3">
                R Spectrum
            </h5>

            <iframe 
                src="spec_2d_M0416_{FILTER}_ID{ID}_R.pdf"
                width="100%"
                height="400px"
                class="rounded bg-white">
            </iframe>

        </div>

        <!-- C SPECTRUM -->

        <div class="col-md-6">

            <h5 class="text-center mb-3">
                C Spectrum
            </h5>

            <iframe 
                src="spec_2d_M0416_{FILTER}_ID{ID}_C.pdf"
                width="100%"
                height="400px"
                class="rounded bg-white">
            </iframe>

        </div>

    </div>

    <!-- GEKO -->

    <h2 class="mb-4">
        GEKO Results
    </h2>

    <div class="bg-secondary rounded p-5 mb-5 text-center">

        {geko_block}

    </div>

    <!-- DINGO -->

    <h2 class="mb-4">
        DINGO Results
    </h2>

    <div class="bg-secondary rounded p-5 text-center">

        {dingo_block}

    </div>

</div>

</body>

</html>
"""

    with open(f"{SOURCE_DIRECTORY}/index.html", "w") as f:
        f.write(html)

    # HOMEPAGE CARD -- gk-card class + data-id/data-filter/data-line/data-geko/data-dingo
    # power the search bar and filter chips on the homepage (see index.html's <script>
    # block). The GEKO/DINGO checkmarks below reflect has_geko/has_dingo, not a hardcoded
    # "both done" -- most sources only have one or the other at any given time.
    card = f"""
        <div class="col-md-4 gk-card" data-id="{ID}" data-filter="{FILTER}" data-line="{line}" data-geko="{'yes' if has_geko else 'no'}" data-dingo="{'yes' if has_dingo else 'no'}">

            <a href="sources/{ID}/index.html"
               class="text-decoration-none text-light">

                <div class="card bg-secondary border-0 shadow h-100">

                    <img src="sources/{ID}/{ID}_rgb_rot.png"
                         class="card-img-top">

                    <div class="card-body">

                        <h4 class="card-title">
                            Source {ID}
                        </h4>

                        <p class="mb-1">
                            z = {zspec:.3f}
                        </p>

                        <p class="mb-1">
                            {FILTER} · {line}
                        </p>

                        <p class="mb-0">
                            GEKO {'✓' if has_geko else '—'} &nbsp;&nbsp; DINGO {'✓' if has_dingo else '—'}
                        </p>

                    </div>

                </div>

            </a>

        </div>
"""

    with open(INDEX_PATH, "r") as f:
        content = f.read()

    if f"Source {ID}" not in content:

        content = content.replace(
            "<!-- SOURCE CARDS -->",
            f"<!-- SOURCE CARDS -->\n\n{card}"
        )

        with open(INDEX_PATH, "w") as f:
            f.write(content)

        print(f"Added Source {ID} to homepage.")

    else:
        print(f"Source {ID} already exists in homepage.")

    print(f"Done: {ID}")

In [9]:
for source in SOURCES:
    process_source(source)

Added Source 3494 to homepage.
Done: 3494
Added Source 3976 to homepage.
Done: 3976
Added Source 4158 to homepage.
Done: 4158
Added Source 4831 to homepage.
Done: 4831
Added Source 6084 to homepage.
Done: 6084
Added Source 11406 to homepage.
Done: 11406
Added Source 11733 to homepage.
Done: 11733
Added Source 12086 to homepage.
Done: 12086
Added Source 12277 to homepage.
Done: 12277
Added Source 12722 to homepage.
Done: 12722
Added Source 14666 to homepage.
Done: 14666
Added Source 15938 to homepage.
Done: 15938
Added Source 15941 to homepage.
Done: 15941
Added Source 18212 to homepage.
Done: 18212
Added Source 18944 to homepage.
Done: 18944
Added Source 20219 to homepage.
Done: 20219
Added Source 22835 to homepage.
Done: 22835
Added Source 23955 to homepage.
Done: 23955
Added Source 30563 to homepage.
Done: 30563
Added Source 34403 to homepage.
Done: 34403
Added Source 36154 to homepage.
Done: 36154


## Diagnostics: convergence check (divergences, r_hat)

Not part of the website pipeline above -- just a quick per-source sanity check on whatever
is currently sitting in the newest batchN under `GEKO_BATCH_ROOTS`, so bad fits can be
spotted before deciding what to add to `SOURCES` / resubmit. Same r_hat/divergence check as
`~/kinematics/kinematics_analysis.ipynb`, but reads straight from `find_latest_batch_run_dir()`
instead of a single hardcoded run directory, so it keeps working as batch5, batch6, ... show up.

In [10]:
import arviz as az
from astropy.table import Table

DIAG_VAR_NAMES = ["PA", "i", "Va", "r_t", "sigma0", "r_eff", "n"]


def load_batch_diagnostics(ids, var_names=DIAG_VAR_NAMES):
    """Per-ID: locate the newest batchN run (find_latest_batch_run_dir), load its
    <ID>_output netcdf, and report divergences + max r_hat across var_names.
    r_hat should be ~1.00-1.01 and divergences should be 0 -- flagged rows ('CHECK')
    mean that source's posterior isn't trustworthy yet."""
    rows = []
    for sid in ids:
        row = {"ID": sid, "status": "ok", "n_divergences": None, "max_rhat": None, "batch_dir": None}

        run_dir = find_latest_batch_run_dir(sid)
        if run_dir is None:
            row["status"] = "not found in any batch"
            rows.append(row)
            continue
        row["batch_dir"] = run_dir

        out_files = glob.glob(f"{run_dir}/**/{sid}_output", recursive=True)
        if not out_files:
            row["status"] = "no _output netcdf"
            rows.append(row)
            continue

        idata = az.from_netcdf(out_files[0])
        n_div = int(idata.sample_stats["diverging"].sum().values)
        summary = az.summary(idata, var_names=var_names)
        max_rhat = float(summary["r_hat"].max())

        row["n_divergences"] = n_div
        row["max_rhat"] = round(max_rhat, 4)
        row["status"] = "ok" if (n_div == 0 and max_rhat <= 1.05) else "CHECK"
        rows.append(row)

    return Table(rows=rows, names=list(rows[0].keys()))


# Defaults to every Ha source currently in SOURCES -- pass your own list of IDs instead
# to check a batch before adding it to SOURCES (e.g. a fresh batch5 you haven't reviewed yet).
diag_ids = [s["id"] for s in SOURCES if s["line"] == "Ha"]
diag_table = load_batch_diagnostics(diag_ids)
diag_table.pprint_all()

print("\nFlagged (divergences > 0 or max_rhat > 1.05, or missing):")
for row in diag_table:
    if row["status"] != "ok":
        print(f"  ID {row['ID']}: {row['status']}  "
              f"(n_div={row['n_divergences']}, max_rhat={row['max_rhat']})")

  ID  status n_divergences max_rhat                                         batch_dir                                         
----- ------ ------------- -------- ------------------------------------------------------------------------------------------
 3494     ok             0      1.0  /xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissions/batch4/3494
 3976     ok             0      1.0  /xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissions/batch4/3976
 4158     ok             0      1.0  /xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissions/batch4/4158
 4831     ok             0      1.0  /xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissions/batch4/4831
 6084  CHECK             0      2.0  /xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissions/batch4/6084
11406     ok             0      1.0 /xdisk/egami/aakhtarkavan/kinematics_backups/july_clean_env_batch_submissio